***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](2_0_introduction.ipynb)
    * 上一节：[2.14 补充专题：一维 CLEAN 的数学演示](2_14_CLEAN_in_1D.ipynb)
    * 下一节：[2.x 延伸阅读与参考文献](2_x_further_reading_and_references.ipynb)

***


## 2.15 统计、不确定度、正则化与过拟合

前面各节建立了傅里叶变换、采样、线性代数和最小二乘的基本语言。进入真实射电干涉数据处理后，还需要另一组同样基础的概念：统计不确定度、协方差、条件数、正则化和过拟合。没有这些概念，许多实践判断只能停留在经验层面。例如，校准解为什么会随参考天线和模型改变，图像噪声为什么不能只用像素均方根表示，自校准为什么可能改善残差却扭曲通量，CLEAN 为什么不能无限迭代到残差消失，源表为什么需要完整性和可靠性验证。

统计语言的作用，是把“看起来不错”变成“证据足以支持”。射电干涉测量中的许多结论都不是直接观测量，而是由带噪可见度经过校准、成像、反卷积、源搜索和测量得到的函数。输入的噪声、权重、模型假设和处理自由度会沿着这条链传播到最终科学量。本节的目标，是给后续第 8 章校准可信度和第 9 章误差预算提供数学底座。


![不确定度传播主线](figures/math_stats_uncertainty_propagation.png)

图 2.15.1 展示了不确定度传播的主线。观测量有均值和方差，模型参数有协方差矩阵，函数关系由雅可比矩阵近似，最终科学量的不确定度可以表现为误差椭圆。这里的重点不是记住某个孤立公式，而是建立从输入噪声到输出结论的传递意识。


### 2.15.1 随机变量、估计量与误差

设观测数据写成

$$
\boldsymbol d = \boldsymbol m(\boldsymbol\theta) + \boldsymbol n,
$$

其中 $\boldsymbol m(\boldsymbol\theta)$ 是由参数 $\boldsymbol\theta$ 给出的模型，$\boldsymbol n$ 是噪声。若噪声均值为零，协方差为 $\mathbf C_d$，则加权最小二乘中常用的目标函数为

$$
\chi^2(\boldsymbol\theta)=
[\boldsymbol d-\boldsymbol m(\boldsymbol\theta)]^{\rm T}
\mathbf C_d^{-1}
[\boldsymbol d-\boldsymbol m(\boldsymbol\theta)].
$$

这个式子说明，误差不是只由残差大小决定，还由残差落在哪个噪声方向上决定。若某些通道、基线或时间段噪声较大，它们在目标函数中应有较小权重；若噪声相关，协方差矩阵不再是对角阵，简单像素计数或通道计数会低估不确定度。射电图像中的相关噪声、谱线 cube 中 Hanning smoothing 后的通道相关、宽带成像中频率项之间的相关，都属于这种情况。

估计量 $\hat{\boldsymbol\theta}$ 是由数据计算出的参数值。估计量的误差包含随机误差和系统误差。随机误差来自噪声实现，通常可以用方差或协方差描述；系统误差来自模型不完备、标尺偏差、主波束模型、缺短间距、校准转移或处理选择。真实报告中只给随机误差通常不够，因为许多射电科学量的主要限制正是系统项。


### 2.15.2 协方差与误差椭圆

对多参数问题，单个误差条无法表达参数之间的相关性。协方差矩阵

$$
\mathbf C_\theta = \left\langle
(\boldsymbol\theta-\langle\boldsymbol\theta\rangle)
(\boldsymbol\theta-\langle\boldsymbol\theta\rangle)^{\rm T}
\right\rangle
$$

同时描述误差大小和方向。对两个参数而言，协方差矩阵的特征向量给出误差椭圆主轴方向，特征值给出主轴长度的平方。若两个参数高度相关，误差椭圆会被拉长，表示某些组合被数据约束得很好，而另一些组合很差。


![协方差误差椭圆](figures/math_stats_covariance_ellipse.png)

图 2.15.2 展示了协方差椭圆。椭圆大小表示不确定度，倾斜方向表示参数相关性。校准增益、天空模型、通量标尺和方向相关自由度常常不是独立参数；忽略协方差会使误差预算显得过于乐观。


在线性近似下，若科学量 $\boldsymbol y$ 是参数的函数 $\boldsymbol y=\boldsymbol f(\boldsymbol\theta)$，在某个解附近有雅可比矩阵

$$
\mathbf J_{ij}=\frac{\partial f_i}{\partial \theta_j},
$$

则输出协方差可近似写成

$$
\mathbf C_y \simeq \mathbf J\mathbf C_\theta \mathbf J^{\rm T}.
$$

这就是误差传播的核心形式。它解释了为什么相同的参数误差对不同科学量影响不同：某些科学量对通量标尺敏感，某些对背景选择敏感，某些对主波束模型敏感，某些对短间距缺失敏感。第 9 章的通量、上限、源表可靠性和限制声明，本质上都在使用这种思想。


### 2.15.3 条件数与病态问题

线性最小二乘问题可写成

$$
\hat{\boldsymbol\theta}=\arg\min_{\boldsymbol\theta}\|\mathbf A\boldsymbol\theta-\boldsymbol d\|^2.
$$

若矩阵 $\mathbf A$ 的某些列几乎线性相关，或者某些奇异值很小，问题就会变得病态。条件数

$$
\kappa(\mathbf A)=\frac{\sigma_{\max}}{\sigma_{\min}}
$$

衡量最大奇异值与最小奇异值之比。条件数大意味着小的数据扰动可能导致大的参数变化。校准中的退化、宽场方向相关解的自由度过多、稀疏 $uv$ 覆盖下的反卷积、谱线多分量拟合，都可能形成病态问题。

病态并不意味着无法处理，而意味着解必须带着约束、先验或限制声明。若数据不能区分两个参数组合，报告不应给出看似精确的单独参数；若某个方向的误差椭圆很长，科学结论应沿着被数据真正约束的组合表达。


### 2.15.4 正则化：限制不被数据支持的自由度

正则化把额外约束写进目标函数。常见形式为

$$
\hat{\boldsymbol\theta}=\arg\min_{\boldsymbol\theta}
\left[\|\mathbf A\boldsymbol\theta-\boldsymbol d\|_{\mathbf C^{-1}}^2
+\lambda\|\mathbf L\boldsymbol\theta\|^2\right],
$$

其中 $\mathbf L$ 可以表示平滑、幅度、曲率或其他先验结构，$\lambda$ 控制数据拟合与模型复杂度之间的折中。$\lambda$ 太小，模型可能追随噪声；$\lambda$ 太大，真实结构会被过度压平。正则化不是为了让图像更漂亮，而是为了明确哪些自由度被数据支持，哪些自由度由先验或处理选择控制。


![正则化折中](figures/math_stats_regularization_tradeoff.png)

图 2.15.3 用折中曲线表示正则化思想。横轴可理解为数据残差，纵轴可理解为模型复杂度。合适区域不是残差最小或模型最简单的极端，而是在证据和自由度之间取得可解释平衡。


在射电成像中，CLEAN mask、多尺度尺度集、停止阈值、最大熵约束、压缩感知稀疏性、方向相关解的平滑约束，都可视为广义正则化。它们决定哪些结构允许进入模型，哪些结构留在残差中。因此，参数文件和限制声明必须记录这些选择。否则，图像中的某个结构究竟来自数据、先验还是算法自由度，就无法复查。


### 2.15.5 过拟合：残差变小不等于结论更真

过拟合指模型自由度超过数据能够支持的程度，使模型开始解释噪声、未建模系统误差或处理伪影。最小二乘残差降低并不自动意味着科学结论更可靠。自校准中，过短解间隔可能把天空模型不完备吸收到增益；CLEAN 中，过深迭代可能把噪声峰纳入模型；源表检测中，过低阈值会提高候选数量却降低可靠性；Faraday 分解中，过多分量可能拟合旁瓣而非真实结构。


![过拟合示意](figures/math_stats_overfitting.png)

图 2.15.4 展示了过拟合的基本形态。复杂模型可以通过带噪数据点，却失去对真实趋势的稳定描述。射电干涉测量中的过拟合往往不以曲线形式出现，而表现为校准解异常灵活、残差被不合理吸收、源表伪检增加或图像结构依赖处理参数。


判断过拟合通常需要交叉证据。校准解应与物理时间尺度相符，图像结构应在合理参数变化下稳定，源表应通过注入恢复和人工检查，谱线结构应在不同 mask 或平滑尺度下保持主要结论，偏振分量应与 RMSF 和频率覆盖相容。若结论只在某个狭窄参数选择下出现，它应被写成候选或限制，而不是直接进入物理解释。


### 2.15.6 本节结论

统计、不确定度、正则化与过拟合把数学工具箱连接到真实数据判断。协方差说明误差方向，条件数说明解的稳定性，正则化说明模型自由度如何被限制，过拟合说明残差降低为何不等于科学结论更可靠。这些概念会在第 8 章的校准解可信度、第 9 章的图像测量、源表验证、归档复盘和处理报告中反复出现。
